# Noise Analysis

Identifies CW noise frequencies and compares filter strategies.

1. **FFT** — mean amplitude spectrum of noise events per channel; reveals CW peaks
2. **PSD (Noise vs Signal)** — Welch PSD of noise and signal event populations
3. **Filter Comparison** — notch vs lowpass applied to example waveforms
4. **RMS Noise** — per-channel RMS for raw, notch, lowpass, and bandpass filtered data

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib ipympl

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import scipy.signal
from pathlib import Path

from naludaq.board import Board

from toolbox.filters.analysis_pipeline import AnalysisPipeline
from toolbox.filters.steps import rollover_step, spike_step, signal_noise_step, noise_step, psd_step
from toolbox.plotting.save_plots import save_plot
from toolbox.plotting.interactive_plot import interactive_plot

## Configuration

| Variable | Description |
|---|---|
| `BOARD` | `"trbhm"` or `"asoc"` — must match the preprocessing checkpoint |
| `CHUNK_SIZE` | Events per chunk; `None` = all at once |
| `ROLLOVER_THRESHOLD` | ADC samples below this are treated as rollover |
| `SIGNAL_THRESHOLD` | ADC threshold for classifying a signal channel as fired |
| `TRIG_THRESHOLD` | ADC threshold for classifying the trigger channel as fired |
| `CW_NOISE_FREQS` | Notch filter frequencies (Hz) — identify from FFT/PSD cells below |
| `LOWPASS_CUTOFF` | Cutoff frequency (Hz) for the lowpass Butterworth filter |
| `BANDPASS_FREQS` | `(low, high)` band in Hz for the bandpass Butterworth filter |
| `N_EXAMPLE_EVENTS` | Number of waveforms shown in the filter comparison plots |

In [ ]:
BOARD = "trbhm"  # "trbhm" or "asoc"

CHUNK_SIZE = None

ROLLOVER_THRESHOLD = -2000
SIGNAL_THRESHOLD   =  100   # signal channels
TRIG_THRESHOLD     = -50   # trigger channel
IS_SIG_RISING_EDGE = True
IS_TRIG_RISING_EDGE = False

# CW peaks to notch -- identify from the FFT / PSD plots below, then fill in
CW_NOISE_FREQS = [1.1718750e9, 1.3281250e9, 1.5625000e9]  # Hz

LOWPASS_CUTOFF  = 1.0e9
BANDPASS_FREQS  = (0.1e9, 1.0e9)  # (low, high) Hz
BUTTER_ORDER    = 3
NOTCH_Q         = 10

N_EXAMPLE_EVENTS = 1

# Derived
CHANNELS        = list(range(8))
SIGNAL_CHANNELS = list(range(7))
TRIGGER_CHANNEL = 7

board         = Board("trbhm") if BOARD == "trbhm" else Board("aodsoc_asoc")
SAMPLING_RATE = board.params["samplingrate"] * 1e9  # Hz

CHECKPOINT_FOLDER = Path().cwd() / "data"
CHECKPOINT_FILE   = f"{BOARD}-preprocessed"

PLOTS_DIR = Path("plots") / "noise"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Board: {BOARD}  |  Sampling rate: {SAMPLING_RATE/1e9:.3f} GHz")

## Load Checkpoint

In [ ]:
def load_checkpoint(folder: Path, prefix: str) -> dict:
    out = {}
    for f in sorted(folder.glob(f"{prefix}-*.npy")):
        key = f.stem.replace(f"{prefix}-", "")
        try:
            out[key] = np.load(f, mmap_mode="r", allow_pickle=True)
            print(f"Loaded  {key:35s}  {out[key].shape}")
        except Exception:
            out[key] = np.load(f, allow_pickle=True)
            print(f"Loaded  {key:35s}  (pickle)")
    return out

checkpoint = load_checkpoint(CHECKPOINT_FOLDER, CHECKPOINT_FILE)

data = checkpoint["data"]  # (N, C, S)
N, C, S = data.shape
print(f"\nEvents: {N}  Channels: {C}  Samples: {S}")

## Interactive Plot -- Raw

In [ ]:
interactive_plot(data, title=f"{BOARD.upper()} -- Raw")

## Analysis Pipeline

Runs rollover correction, spike removal, signal/noise classification, and the CW notch filter.
`noise_filtered` is the notch output and is used directly in the RMS and filter comparison sections.

In [ ]:
pipe = AnalysisPipeline()

pipe.add(rollover_step,
         min_threshold=ROLLOVER_THRESHOLD, apply_correction=True, correction=4096)

pipe.add(spike_step,
         source_key="rollover_corrected",
         apply_correction=True,
         baseline_lookback_window=2)

pipe.add(signal_noise_step,
         source_key="spike_filtered",
         signal_threshold=SIGNAL_THRESHOLD, trig_threshold=TRIG_THRESHOLD,
         is_sig_rising_edge=IS_SIG_RISING_EDGE, is_trig_rising_edge=IS_TRIG_RISING_EDGE,
         in_place=True, store_waveforms=True)

pipe.add(psd_step, f_s=SAMPLING_RATE)

pipe.add(noise_step,
         source_key="spike_filtered", save_key="noise_filtered",
         f_s=SAMPLING_RATE, filter_type="notch", q_factor=NOTCH_Q, freqs=CW_NOISE_FREQS, in_place=False)

pipe.add(noise_step,
         source_key="spike_filtered", save_key="lowpass_filtered",
         f_s=SAMPLING_RATE, filter_type="lowpass", order=BUTTER_ORDER, freqs=[LOWPASS_CUTOFF], in_place=False)

print(pipe)

In [ ]:
results = pipe.run(data, chunk_size=CHUNK_SIZE)


In [ ]:
spike_filtered    = np.asarray(results["spike_filtered"])     # (N, C, S)
noise_filtered    = np.asarray(results["noise_filtered"])     # (N, C, S) notch-filtered
lowpass_filtered  = np.asarray(results["lowpass_filtered"])   # (N, C, S) lowpass-filtered
noise_data        = np.asarray(results["noise_data"])         # (K, S) noise event waveforms
noise_events      = np.asarray(results["noise_events"])       # (K,) event index per pair
noise_channels    = np.asarray(results["noise_channels"])     # (K,) channel index per pair
signal_events     = np.asarray(results["signal_events"])
signal_channels   = np.asarray(results["signal_channels"])
noise_psd         = np.asarray(results["noise_p"])            # (K, F)
signal_psd        = np.asarray(results["signal_p"])           # (K, F)
noise_freq        = np.asarray(results["noise_freq"])
signal_freq       = np.asarray(results["signal_freq"])
trig_signal_p     = np.asarray(results["trig_signal_p"])
trig_signal_freq  = np.asarray(results["trig_signal_freq"])
trig_noise_p      = np.asarray(results["trig_noise_p"])
trig_noise_freq   = np.asarray(results["trig_noise_freq"])

noise_event_idx  = np.unique(noise_events)
signal_event_idx = np.unique(signal_events)

noise_waveforms  = spike_filtered[noise_event_idx]
signal_waveforms = spike_filtered[signal_event_idx]

print(f"Noise events:  {len(noise_event_idx):>6}")
print(f"Signal events: {len(signal_event_idx):>6}")

---
## FFT -- Identify CW Noise Frequencies

Mean FFT amplitude of **noise events** per channel. Sharp peaks are CW
interference candidates -- note their frequencies and set `CW_NOISE_FREQS` above.

In [ ]:
from scipy.fft import rfft, rfftfreq
SAMPLING_RATE = board.params['samplingrate'] * 1e9 # Hz
FFT_EVENT, FFT_CH = 0, 5


fft_subset = results["spike_filtered"][noise_events[0]][noise_channels[0]]
fft_mag = np.abs(rfft(fft_subset))
fft_freqs = rfftfreq(fft_subset.size, 1/SAMPLING_RATE)
plt.figure()
plt.title(f"FFT of Event {FFT_EVENT} Channel {FFT_CH}")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Mag")
plt.yscale("log")
plt.stem(fft_freqs, fft_mag)
plt.show()

In [ ]:
from matplotlib.lines import Line2D
fig, axes = plt.subplots(1,2, figsize=(20, 10), sharey=True)
titles = ["Channels with Signals of Interest", "Channels with Noise"]

fig.suptitle(f"Power Spectral Density of Non-Trigger Channels - {BOARD.lower()}")
ticks = np.arange(0, max(signal_freq), 0.25e9)

for ax, title in zip(axes, titles):
    ax.set_title(title)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_xticks(ticks)

axes[0].set_ylabel("Counts^2/Hz")

for psd in signal_psd:
    axes[0].plot(signal_freq, psd)

for psd in noise_psd:
    axes[1].plot(noise_freq, psd)

print(signal_psd.shape)

signal_mean_psd = np.mean(signal_psd, axis=0)
signal_peaks, _ = scipy.signal.find_peaks(signal_mean_psd)

signal_top = signal_peaks[np.argsort(signal_mean_psd[signal_peaks])[-8:]]
signal_peak_freqs = signal_freq[signal_top]

signal_handles = [
    Line2D([0], [0], color='none', label=f"{(f)/(1e9):.4f} GHz")
    for f in sorted(signal_peak_freqs)
]

axes[0].legend(handles=signal_handles, title="Top PSD Peaks")



noise_mean_psd = np.mean(noise_psd, axis=0)
print(noise_mean_psd.shape)
noise_peaks, _ = scipy.signal.find_peaks(noise_mean_psd)

noise_top = noise_peaks[np.argsort(noise_mean_psd[noise_peaks])[-8:]]
noise_peak_freqs = noise_freq[noise_top]

noise_handles = [
    Line2D([0], [0], color='none', label=f"{(f)/(1e9):.4f} GHz")
    for f in sorted(noise_peak_freqs)
]

axes[1].legend(handles=noise_handles, title="Top PSD Peaks")


fig.tight_layout()
plt.show()


---
## PSD -- Per-Channel (Non-Trigger)

One row per signal channel; left = signal events, right = noise events.
Top PSD peaks are listed in the legend.

In [ ]:
fig, axes = plt.subplots(7, 2, figsize=(20, 25), sharey=True)
titles = ["Channels with Signals of Interest", "Channels with Noise"]

fig.suptitle(f"Power Spectral Density of Non-Trigger Channels - {BOARD.lower()}")
ticks = np.arange(0, max(signal_freq), 0.25e9)

for ax in axes:
    ax[0].set_xlabel("Frequency Hz")
    ax[1].set_xlabel("Frequency Hz")
    ax[0].set_xticks(ticks)
    ax[1].set_xticks(ticks)
    ax[0].set_ylabel("Counts^2/Hz")
    ax[1].set_ylabel("Counts^2/Hz")

for ch_idx in range(7):
    axes[ch_idx, 0].set_title(f"Channel {ch_idx}")
    ch_signal_p = signal_psd[np.where(signal_channels == ch_idx)]
    ch_noise_p  = noise_psd[np.where(noise_channels == ch_idx)]

    for psd in ch_signal_p:
        axes[ch_idx, 0].plot(signal_freq, psd)
    for psd in ch_noise_p:
        axes[ch_idx, 1].plot(noise_freq, psd)

    if ch_signal_p.shape[0] > 0:
        signal_mean_psd = np.mean(ch_signal_p, axis=0)
        signal_peaks, _ = scipy.signal.find_peaks(signal_mean_psd)
        signal_top = signal_peaks[np.argsort(signal_mean_psd[signal_peaks])[-8:]]
        signal_peak_freqs = signal_freq[signal_top]
        signal_handles = [
            Line2D([0], [0], color='none', label=f"{f/1e9:.4f} GHz")
            for f in sorted(signal_peak_freqs)
        ]
        axes[ch_idx, 0].legend(handles=signal_handles, title="Top PSD Peaks")

    if ch_noise_p.shape[0] > 0:
        noise_mean_psd = np.mean(ch_noise_p, axis=0)
        noise_peaks, _ = scipy.signal.find_peaks(noise_mean_psd)
        noise_top = noise_peaks[np.argsort(noise_mean_psd[noise_peaks])[-8:]]
        noise_peak_freqs = noise_freq[noise_top]
        noise_handles = [
            Line2D([0], [0], color='none', label=f"{f/1e9:.4f} GHz")
            for f in sorted(noise_peak_freqs)
        ]
        axes[ch_idx, 1].legend(handles=noise_handles, title="Top PSD Peaks")

fig.tight_layout()
plt.show()

## PSD -- Trigger Channel

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 10))

fig.suptitle(f"Power Spectral Density of Trigger Channel - {BOARD.upper()}")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Counts^2/Hz")

for psd in trig_signal_p:
    ax.plot(trig_signal_freq, psd)

trig_signal_mean_psd = np.mean(trig_signal_p, axis=0)
trig_signal_peaks, _ = scipy.signal.find_peaks(trig_signal_mean_psd)
trig_signal_top = trig_signal_peaks[np.argsort(trig_signal_mean_psd[trig_signal_peaks])[-9:]]
trig_signal_peak_freqs = trig_signal_freq[trig_signal_top]

trig_handles = [
    Line2D([0], [0], color='none', label=f"{f/1e9:.4f} GHz")
    for f in sorted(trig_signal_peak_freqs)
]
ax.legend(handles=trig_handles, title="Top PSD Peaks")

fig.tight_layout()
plt.show()

---
## Filter Comparison -- Notch vs Lowpass

Overlays raw (spike-corrected), notch-filtered, and lowpass-filtered waveforms
for the same noise event so you can see what each filter removes.

In [ ]:
example_indices = noise_event_idx[:N_EXAMPLE_EVENTS]
t = np.arange(S) / SAMPLING_RATE * 1e9  # ns

for ev_idx in example_indices:
    fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharex=True)
    for i, ch in enumerate(CHANNELS):
        ax = axes[i // 4, i % 4]
        raw     = spike_filtered[ev_idx, ch].astype(float)
        notched = noise_filtered[ev_idx, ch].astype(float)
        lowpsd  = lowpass_filtered[ev_idx, ch].astype(float)
        if not np.any(np.isfinite(raw)):
            ax.set_title(f"Ch {ch} -- all NaN")
            continue
        ax.plot(t, raw,     linewidth=0.8, alpha=0.8, label="raw")
        ax.plot(t, notched, linewidth=0.9, alpha=0.8, label="notch")
        ax.plot(t, lowpsd,  linewidth=0.9, alpha=0.8, label="lowpass")
        ax.set_title(f"Ch {ch}" + (" [trig]" if ch == TRIGGER_CHANNEL else ""))
        ax.set_xlabel("Samples [n]")
        ax.set_ylabel("ADC Counts")
        if i == 0:
            ax.legend(fontsize=8, loc="upper right")
    fig.suptitle(f"{BOARD.upper()} -- Filter Comparison (event {ev_idx})")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    save_plot(fig, PLOTS_DIR / f"{BOARD}-filter-compare-ev{ev_idx}")
    plt.show()

---
## RMS Noise -- Filter Comparison

Per-channel RMS histograms over all **noise (event, channel) pairs** for three cases:

| Case | Description |
|---|---|
| pedestal-corrected | No additional filtering — spike-corrected only |
| notch | IIR notch at each `CW_NOISE_FREQS` frequency |
| lowpass | Butterworth lowpass at `LOWPASS_CUTOFF` |

In [ ]:
noise_after_notch   = (spike_filtered - noise_filtered)[noise_events, noise_channels, :]
noise_after_lowpass = (spike_filtered - lowpass_filtered)[noise_events, noise_channels, :]

rms_notch = np.sqrt(np.nanmean(noise_after_notch**2,   axis=-1))
rms_lp    = np.sqrt(np.nanmean(noise_after_lowpass**2, axis=-1))
rms_raw   = np.sqrt(np.nanmean(noise_data**2,          axis=-1))

In [ ]:
fig, axes = plt.subplots(7, 1, sharex=True, sharey=True, figsize=(10, 15))

fig.suptitle(f"Noise RMS - {BOARD.lower()}")
for ch_idx in range(7):
    axes[ch_idx].hist(rms_notch[np.where(noise_channels == ch_idx)], align="left", label="notch")
    axes[ch_idx].hist(rms_lp[np.where(noise_channels == ch_idx)],    align="left", label="lowpass",           alpha=0.5)
    axes[ch_idx].hist(rms_raw[np.where(noise_channels == ch_idx)],   align="left", label="pedestal-corrected", alpha=0.5)
    axes[ch_idx].set_title(f"Channel {ch_idx}")
    axes[ch_idx].set_xlabel("Noise RMS (ADC Counts)")
    axes[ch_idx].set_ylabel("Occurrences")
    axes[ch_idx].legend(["notch", "lowpass", "pedestal-corrected"], loc="upper right")

fig.tight_layout()
save_plot(fig, PLOTS_DIR / f"{BOARD}-rms-filter-compare")
plt.show()